# Tutorial 4: Performance Optimization and Caching

## What You Will Learn

- Use `@cacheable` to skip redundant computation
- Understand how Scalable hashes function arguments
- Handle file-based inputs with `FileType` and `DirType`
- Force recomputation and invalidate caches
- Monitor cache hit/miss rates

## Prerequisites

- Completed Tutorial 1
- `pip install scalable`

In [ ]:
import os
import tempfile
import time

project_dir = tempfile.mkdtemp(prefix="scalable-caching-")
os.chdir(project_dir)
print(f"Working directory: {project_dir}")

# Set cache dir for this tutorial
os.environ["SCALABLE_CACHE_DIR"] = os.path.join(project_dir, "cache")

## Step 1: Basic @cacheable

The `@cacheable` decorator intercepts function calls, computes a cache key from the function name and arguments, and returns cached results when available.

In [ ]:
from scalable import cacheable


@cacheable(return_type=dict, scenario_id=int)
def expensive_simulation(scenario_id: int) -> dict:
    """Simulates an expensive computation."""
    time.sleep(1)  # Simulate 1 second of work
    return {"scenario": scenario_id, "result": scenario_id * 42}


print("Function defined with @cacheable")
print(f"Cache directory: {os.environ['SCALABLE_CACHE_DIR']}")

In [ ]:
# First call — cache MISS (takes ~1 second)
start = time.time()
result1 = expensive_simulation(42)
elapsed1 = time.time() - start

print(f"First call: {result1}")
print(f"Time: {elapsed1:.3f}s (cache MISS)")

In [ ]:
# Second call — cache HIT (instant)
start = time.time()
result2 = expensive_simulation(42)
elapsed2 = time.time() - start

print(f"Second call: {result2}")
print(f"Time: {elapsed2:.3f}s (cache HIT)")
print(f"\nSpeedup: {elapsed1/max(elapsed2, 0.001):.0f}x")

In [ ]:
# Different argument — cache MISS
start = time.time()
result3 = expensive_simulation(99)
elapsed3 = time.time() - start

print(f"New argument: {result3}")
print(f"Time: {elapsed3:.3f}s (cache MISS — different key)")

### How It Works

1. Arguments are serialized with `dill` and hashed with `xxhash` (seeded by `SCALABLE_SEED`)
2. Function name + argument hash = composite cache key
3. On hit: stored result is deserialized and returned
4. On miss: function executes, result is stored

## Step 2: Type Annotations for Reliable Hashing

Explicit types ensure deterministic cache keys across Python type variations.

In [ ]:
@cacheable(return_type=str, name=str, count=int)
def greet(name: str, count: int) -> str:
    """Generate a greeting."""
    return f"Hello {name}! (x{count})"


# Consistent hashing regardless of source type
r1 = greet("World", 3)
r2 = greet("World", 3)  # Cache hit
print(f"Result: {r1}")
print(f"Same result from cache: {r1 == r2}")

## Step 3: Hashing Files and Directories

Scientific workflows often operate on input files. `FileType` hashes file **content** (not paths).

In [ ]:
from scalable import cacheable, FileType

# Create a sample input file
with open("input_data.csv", "w") as f:
    f.write("scenario,temperature,precipitation\n")
    f.write("1,300,1200\n")
    f.write("2,310,1100\n")

print("Created input_data.csv")

In [ ]:
@cacheable(return_type=dict, data_file=FileType)
def process_data(data_file: str) -> dict:
    """Process a data file. Cache key includes file contents."""
    time.sleep(0.5)
    with open(data_file) as f:
        lines = f.readlines()
    return {"records": len(lines) - 1, "file": data_file}


# First call — hashes file content
start = time.time()
r1 = process_data("input_data.csv")
print(f"First call: {r1} ({time.time()-start:.3f}s)")

# Second call — same file content = cache hit
start = time.time()
r2 = process_data("input_data.csv")
print(f"Second call: {r2} ({time.time()-start:.3f}s) — HIT")

In [ ]:
# Modify the file — cache miss (content changed)
with open("input_data.csv", "a") as f:
    f.write("3,305,1300\n")

start = time.time()
r3 = process_data("input_data.csv")
print(f"After modification: {r3} ({time.time()-start:.3f}s) — MISS (content changed)")

### Type Hashing Strategies

| Type | Strategy |
|------|----------|
| `FileType` | Streams file in 1MB chunks through xxhash. Includes basename. |
| `DirType` | Walks directory, hashes each file path + content (sorted). |
| `str` | Hashes UTF-8 bytes directly. |
| `int` | Hashes byte representation. |

## Step 4: Forcing Recomputation

Use `recompute=True` to invalidate a specific function's cache (e.g., after fixing a bug).

In [ ]:
@cacheable(return_type=dict, recompute=True, scenario_id=int)
def fixed_simulation(scenario_id: int) -> dict:
    """Always recomputes — ignores cache."""
    return {"scenario": scenario_id, "result": scenario_id * 1.7}  # Fixed formula


# Both calls execute the function (recompute=True)
r1 = fixed_simulation(42)
r2 = fixed_simulation(42)
print(f"Call 1: {r1}")
print(f"Call 2: {r2}")
print("Both computed fresh (recompute=True bypasses cache reads)")

## Step 5: Minimal @cacheable Form

For quick prototyping, `@cacheable` works without explicit types:

In [ ]:
@cacheable
def quick_add(x, y):
    """Minimal cacheable form — no explicit types."""
    time.sleep(0.3)
    return x + y


start = time.time()
r1 = quick_add(10, 20)
print(f"First: {r1} ({time.time()-start:.3f}s)")

start = time.time()
r2 = quick_add(10, 20)
print(f"Second: {r2} ({time.time()-start:.3f}s) — cache hit")

> **Recommendation:** Always use explicit types for production code. The minimal form is acceptable for experiments where cache key stability isn't critical.

## Step 6: Cache Directory Structure

In [ ]:
from pathlib import Path

cache_dir = Path(os.environ["SCALABLE_CACHE_DIR"])

if cache_dir.exists():
    print(f"Cache directory: {cache_dir}")
    total_size = sum(f.stat().st_size for f in cache_dir.rglob("*") if f.is_file())
    file_count = sum(1 for f in cache_dir.rglob("*") if f.is_file())
    print(f"Files: {file_count}")
    print(f"Total size: {total_size / 1024:.1f} KB")
else:
    print("Cache directory not yet created")

## Step 7: Cache Invalidation Strategies

| Strategy | Scope | When to Use |
|----------|-------|-------------|
| `recompute=True` | Single function | After fixing a bug in one function |
| Change `SCALABLE_SEED` | Global | After major refactoring |
| Version the function name | Single function | Between known-breaking changes |
| Delete cache directory | Global | Nuclear option |

In [ ]:
# Strategy: Version the function name
@cacheable(return_type=dict, scenario_id=int)
def run_model_v2(scenario_id: int) -> dict:
    """v2 has a different cache key than v1 by virtue of its name."""
    return {"scenario": scenario_id, "result": scenario_id * 3.14}


r = run_model_v2(42)
print(f"v2 result: {r}")
print("v1 cache entries remain but are now 'stale' (different function name)")

## Step 8: Caching in a Distributed Workflow

Combine `@cacheable` with the Session API for production-grade caching.

In [ ]:
from scalable import ScalableSession, cacheable

# Write manifest
manifest = """\
version: 1
project:
  name: caching-demo
targets:
  local:
    provider: local
    max_workers: 2
    threads_per_worker: 1
    processes: false
    containers: none
components:
  worker:
    cpus: 1
    memory: 1G
tasks:
  compute:
    component: worker
    cache: true
"""

with open("scalable.yaml", "w") as f:
    f.write(manifest)


@cacheable(return_type=dict, n=int)
def cached_task(n: int) -> dict:
    """A cacheable distributed task."""
    time.sleep(0.5)
    return {"n": n, "square": n**2}


print("Ready for distributed caching demo")

In [ ]:
# Run 1: All cache misses
session = ScalableSession.from_yaml("./scalable.yaml", target="local")
client = session.start()

start = time.time()
futures = [client.submit(cached_task, i, tag="worker") for i in range(6)]
results = client.gather(futures)
run1_time = time.time() - start

print(f"Run 1 (all misses): {run1_time:.2f}s")
print(f"Results: {results[:3]}...")
session.close()

In [ ]:
# Run 2: All cache hits (same arguments)
session = ScalableSession.from_yaml("./scalable.yaml", target="local")
client = session.start()

start = time.time()
futures = [client.submit(cached_task, i, tag="worker") for i in range(6)]
results = client.gather(futures)
run2_time = time.time() - start

print(f"Run 2 (all hits): {run2_time:.2f}s")
print(f"Speedup: {run1_time/max(run2_time, 0.01):.1f}x")
session.close()

## Summary

1. `@cacheable` prevents redundant computation across retries
2. Explicit type annotations ensure stable cache keys
3. `FileType`/`DirType` hash file **content**, not paths
4. `recompute=True` forces fresh execution for debugging
5. Cache + distributed = major time savings on repeated runs

## Next Steps

- **Tutorial 5**: Remote cache sharing with S3/GCS
- **Tutorial 6**: Monitor cache hit rates in telemetry
- **Tutorial 7**: Handle cache corruption gracefully

In [ ]:
import shutil
os.chdir("/tmp")
shutil.rmtree(project_dir, ignore_errors=True)
if "SCALABLE_CACHE_DIR" in os.environ:
    del os.environ["SCALABLE_CACHE_DIR"]
print("Cleaned up.")